# 01 - Geracao de Mascaras e Segmentacao

## Imports

imports do projeto

In [ ]:
import sys
from pathlib import Path

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import gc
import os
import time

import cv2
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter, binary_opening

from src.io.indice_loader import carregar_indice_excel
from src.io.path_utils import caminho_foto_original, caminho_ground_truth, caminho_ground_truth_output, caminho_segmentada_modelo
from rembg import new_session, remove
from src.runtime.runtime_config import ORT_PROVIDERS, available_providers, resolver_providers, use_cuda
from src.logs.segmentacao_logs import imprimir_resumo_modelo, imprimir_status

if use_cuda:
    print('GPU NVIDIA ativa: usando CUDAExecutionProvider com fallback para CPU.')
else:
    print('GPU NVIDIA/CUDA indisponivel: usando somente CPUExecutionProvider.')

print(f'Providers disponiveis no ambiente: {available_providers}')
print(f'Providers configurados para a sessao: {ORT_PROVIDERS}')


## Configuração

In [ ]:
from src.config import (
    GENERATED_DIR,
    GROUND_OF_TRUTH_OUTPUT,
    MODELOS_PARA_AVALIACAO,
    SEGMENTED_PHOTOS_DIR,
)

## Leitura do índice


In [ ]:
indice_excel = carregar_indice_excel()

## Verificação de integridade dos PNGs gerados

Esta célula valida todos os arquivos .png dentro de `generated/` (incluindo subpastas). Se um arquivo estiver corrompido ou não puder ser aberto corretamente, ele é removido.

In [ ]:
# Verifica integridade dos PNGs em generated/ e remove arquivos corrompidos
def verificar_e_limpar_pngs_corrompidos(diretorio_base):
    total_png = 0
    arquivos_integros = 0
    arquivos_removidos = 0
    falhas_remocao = 0

    for raiz, _, arquivos in os.walk(diretorio_base):
        for nome_arquivo in arquivos:
            if not nome_arquivo.lower().endswith('.png'):
                continue

            total_png += 1
            caminho_arquivo = os.path.join(raiz, nome_arquivo)

            try:
                with Image.open(caminho_arquivo) as img:
                    img.verify()

                with Image.open(caminho_arquivo) as img:
                    img.load()

                arquivos_integros += 1
            except Exception as e:
                print(f"Corrompido detectado: {caminho_arquivo}")
                print(f" - Erro: {e}")

                try:
                    os.remove(caminho_arquivo)
                    arquivos_removidos += 1
                    print(" - Arquivo removido com sucesso.")
                except OSError as erro_remocao:
                    falhas_remocao += 1
                    print(f" - Falha ao remover arquivo: {erro_remocao}")

    print('\nVerificacao de integridade concluida.')
    print(f" - Total de PNGs verificados: {total_png}")
    print(f" - Arquivos integros: {arquivos_integros}")
    print(f" - Arquivos removidos por corrupcao: {arquivos_removidos}")
    print(f" - Falhas ao remover: {falhas_remocao}")

verificar_e_limpar_pngs_corrompidos(GENERATED_DIR)

## Conversão do Ground Truth para PNG Binarizado

Esta seção implementa a conversão das máscaras Ground Truth (JPG → PNG binário) utilizando o **método cientificamente validado** através de análise comparativa.

### Problema: Artefatos de Compressão JPG

O formato JPG utiliza compressão com perdas que gera **artefatos nas bordas** (pixels cinzas intermediários onde deveria haver transição abrupta preto/branco). Ao aplicar um limiar simples (threshold), esses artefatos causam **bordas serrilhadas** que aumentam artificialmente o **perímetro** calculado, prejudicando análises morfométricas.

### Método Selecionado: Blur Gaussiano + Morphological Opening

Baseado em análise científica comparativa (notebook `selecao_metodo_binarizacao.ipynb`), o método **Gaussian Blur + Morphological Opening** foi selecionado como padrão de produção com **pontuação 96.56/100**.

**Critérios de avaliação** (387 imagens):
- **Consistência** (40%): Coeficiente de variação entre máscaras - mede estabilidade do método
- **Circularidade** (30%): Proximidade à forma circular ideal (4πA/P²) - detecta distorções
- **Suavidade de bordas** (30%): Desvio padrão da curvatura - mede qualidade das bordas

**Por que este método venceu:**
- Melhor equilíbrio entre remoção de ruído e preservação de detalhes anatômicos
- Bordas mais suaves (100/100 em suavidade)
- Forma mais consistente (100/100 em circularidade)
- Alta estabilidade entre imagens (91.40/100 em consistência)

**Parâmetros utilizados:**
- `sigma=1.0`: Intensidade do blur gaussiano (suaviza artefatos JPG)
- `threshold=127`: Limiar de binarização (ponto médio da escala 0-255)
- `kernel_size=3`: Tamanho do elemento estruturante para opening (remove pixels isolados)

**Comparação com métodos rejeitados:**
- Limiar simples (0.00/100): Bordas muito serrilhadas, alta variabilidade
- Blur gaussiano (74.04/100): Bom mas sem remoção de ruído isolado
- Filtro bilateral (74.81/100): Preserva bordas mas mais lento e menos consistente


### Implementação

**Pipeline de processamento:**
1. Carrega máscara JPG original
2. Converte para escala de cinza (8-bit grayscale)
3. Aplica **Gaussian Blur** (sigma=1.0) para suavizar artefatos de compressão
4. Aplica **threshold** em 127 para binarização (0 ou 255)
5. Aplica **Morphological Opening** (erosão + dilatação) para remover pixels isolados
6. Salva resultado como PNG binário

**Output:** As máscaras binarizadas são salvas em `generated/Ground-of-truth/*.png`

**Detalhes técnicos:**
- **Gaussian Filter** (`scipy.ndimage.gaussian_filter`): Remove ruído de alta frequência (artefatos JPG)
- **Binary Opening** (`scipy.ndimage.binary_opening`): Remove pequenos objetos e suaviza contornos
- **Elemento estruturante**: Matriz 3×3 de valores True (conectividade completa)


In [ ]:
# ===== PARÂMETROS =====
SIGMA = 1.0
LIMIAR = 127
KERNEL_SIZE = 3

# ===== CONVERSÃO =====
output_dir = GROUND_OF_TRUTH_OUTPUT
os.makedirs(output_dir, exist_ok=True)

print("Convertendo Ground Truth para PNG binarizado")
print(f"Método: Gaussian Blur + Morphological Opening")
print(f"Parâmetros: SIGMA={SIGMA}, LIMIAR={LIMIAR}, KERNEL_SIZE={KERNEL_SIZE}")
print(f"Total de imagens: {len(indice_excel)}\n")

processadas = 0
puladas = 0
erros = 0

for idx, linha in enumerate(indice_excel, 1):
    caminho_original = caminho_ground_truth(linha.nome_arquivo)
    caminho_saida = os.path.join(output_dir, f"{linha.nome_arquivo}.png")
    
    if not os.path.isfile(caminho_original):
        print(f"[{idx}/{len(indice_excel)}] ERRO: Máscara não encontrada: {caminho_original}")
        erros += 1
        continue
    
    if os.path.isfile(caminho_saida):
        puladas += 1
        continue
    
    try:
        with Image.open(caminho_original) as img:
            img_gray = img.convert('L')
            matriz = np.array(img_gray, dtype=np.float32)
            
            # 1. Aplica filtro gaussiano
            matriz_blur = gaussian_filter(matriz, sigma=SIGMA)
            
            # 2. Binariza
            matriz_binaria = (matriz_blur > LIMIAR).astype(bool)
            
            # 3. Morphological opening (remove ruído pequeno)
            estrutura = np.ones((KERNEL_SIZE, KERNEL_SIZE), dtype=bool)
            matriz_opening = binary_opening(matriz_binaria, structure=estrutura)
            
            # 4. Converte para uint8
            matriz_final = (matriz_opening * 255).astype(np.uint8)
            img_binarizada = Image.fromarray(matriz_final, mode='L')
            img_binarizada.save(caminho_saida, format='PNG')
        
        processadas += 1
        if processadas % 10 == 0 or processadas == len(indice_excel):
            print(f"[{idx}/{len(indice_excel)}] Processadas: {processadas} | Puladas: {puladas}")
    
    except Exception as e:
        print(f"[{idx}/{len(indice_excel)}] ERRO ao processar {linha.nome_arquivo}: {e}")
        erros += 1

print(f"\n{'='*60}")
print("Conversão CONCLUÍDA")
print(f"Processadas: {processadas} | Puladas: {puladas} | Erros: {erros}")
print(f"Salvo em: {output_dir}")
print(f"{'='*60}")


## Processa as imagens e gera a máscara binária

In [ ]:
def log_status(stats_geral, stats_modelo, nome_modelo):
    tempo_geral = time.perf_counter() - stats_geral['inicio']
    tempo_modelo = time.perf_counter() - stats_modelo['inicio']
    imprimir_status(stats_geral, stats_modelo, nome_modelo, tempo_geral, tempo_modelo)


total_modelos = len(MODELOS_PARA_AVALIACAO)
total_imagens = len(indice_excel)
total_previsto = total_modelos * total_imagens

stats_geral = {
    'total': total_previsto,
    'processadas': 0,
    'ok': 0,
    'skip': 0,
    'erro': 0,
    'inicio': time.perf_counter(),
    'tempo_inferencia': 0.0
}

for model, provider_config in MODELOS_PARA_AVALIACAO.items():
    providers = resolver_providers(provider_config, model)
    print(f'Iniciando modelo: {model} (provider: {provider_config})')

    stats_modelo = {
        'total': total_imagens,
        'processadas': 0,
        'ok': 0,
        'skip': 0,
        'erro': 0,
        'inicio': time.perf_counter(),
        'tempo_inferencia': 0.0
    }

    try:
        rembg_session = new_session(model, providers=providers)
    except Exception as e:
        print(f" - Falha ao iniciar sessao com providers {providers}: {e}")
        print(' - Recriando sessao em CPU...')
        rembg_session = new_session(model, providers=['CPUExecutionProvider'])

    output_dir = os.path.join(SEGMENTED_PHOTOS_DIR, model)
    os.makedirs(output_dir, exist_ok=True)

    for linha in indice_excel:
        original_path = caminho_foto_original(linha.nome_arquivo)
        mascara_path = caminho_ground_truth(linha.nome_arquivo)
        output_path = caminho_segmentada_modelo(model, linha.nome_arquivo)

        if not os.path.isfile(original_path):
            print(f"[ERRO ] Arquivo original nao encontrado: {original_path}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            log_status(stats_geral, stats_modelo, model)
            continue

        if not os.path.isfile(mascara_path):
            print(f"[ERRO ] Arquivo de mascara nao encontrado: {mascara_path}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            log_status(stats_geral, stats_modelo, model)
            continue

        if os.path.isfile(output_path):
            stats_geral['processadas'] += 1
            stats_geral['skip'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['skip'] += 1
            log_status(stats_geral, stats_modelo, model)
            continue

        try:
            inicio_inferencia = time.perf_counter()
            with Image.open(original_path) as input_rembg:
                output_rembg = remove(
                    input_rembg,
                    only_mask=True,
                    post_process_mask=True,
                    session=rembg_session
                )
            output_rembg.save(output_path)

            duracao_inferencia = time.perf_counter() - inicio_inferencia
            stats_geral['processadas'] += 1
            stats_geral['ok'] += 1
            stats_geral['tempo_inferencia'] += duracao_inferencia

            stats_modelo['processadas'] += 1
            stats_modelo['ok'] += 1
            stats_modelo['tempo_inferencia'] += duracao_inferencia

            log_status(stats_geral, stats_modelo, model)
        except Exception as e:
            print(f"[ERRO ] Falha ao segmentar {linha.nome_arquivo} no modelo {model}: {e}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            log_status(stats_geral, stats_modelo, model)

    tempo_modelo = time.perf_counter() - stats_modelo['inicio']
    imprimir_resumo_modelo(model, stats_modelo, tempo_modelo)

    # Libera memoria da GPU entre modelos
    del rembg_session
    gc.collect()
    if use_cuda:
        # Força liberação de memória CUDA pelo ONNX Runtime
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
        except ImportError:
            pass  # torch não instalado, segue sem limpar cache CUDA